In [1]:
# ==============================================================================
# ENHANCED CONVNEXT-BASE — FULL PIPELINE
# Additions vs previous: EMA, CutMix, 4-TTA, Gradient Clipping,
#                        Stratified Split, Warm Restarts, Better Head
# Estimated runtime: ~110 minutes on Kaggle T4 GPU
# ==============================================================================

import os, glob, random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════
#  1. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")

# ══════════════════════════════════════════════════════════════════
#  2. DYNAMIC PATH DISCOVERY
# ══════════════════════════════════════════════════════════════════
def find_data_paths():
    for root, dirs, _ in os.walk('/kaggle/input'):
        if 'train' in dirs and 'test' in dirs:
            return os.path.join(root, 'train'), os.path.join(root, 'test')
    return None, None

DATA_DIR, TEST_DIR = find_data_paths()
print(f"📂 Train: {DATA_DIR}")
print(f"📂 Test:  {TEST_DIR}")

# ══════════════════════════════════════════════════════════════════
#  3. CONFIGURATION
# ══════════════════════════════════════════════════════════════════
IMG_SIZE    = 288
BATCH_SIZE  = 16
EPOCHS      = 20
SAVE_PATH   = "/kaggle/working/best_model.pth"

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ══════════════════════════════════════════════════════════════════
#  4. AUGMENTATION TRANSFORMS
# ══════════════════════════════════════════════════════════════════
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transforms = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── 4 TTA transforms used at inference ───────────────────────────
tta_flip = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

tta_large = transforms.Compose([
    transforms.Resize(360),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

tta_jitter = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

TTA_TRANSFORMS = [val_transforms, tta_flip, tta_large, tta_jitter]

# ══════════════════════════════════════════════════════════════════
#  5. MODEL ARCHITECTURE
# ══════════════════════════════════════════════════════════════════
class CustomConvNeXt(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        print("✅ Loaded ConvNeXt-Base ImageNet weights")

        # Split backbone into 4 blocks so each can have its own learning rate
        self.block0 = nn.Sequential(*base.features[0:2])
        self.block1 = nn.Sequential(*base.features[2:4])
        self.block2 = nn.Sequential(*base.features[4:6])
        self.block3 = nn.Sequential(*base.features[6:8])

        in_features = base.classifier[2].in_features  # 1024 for ConvNeXt-Base

        # Enhanced head: extra linear layer + BatchNorm + GELU for more capacity
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            base.classifier[0],          # LayerNorm from original
            nn.Flatten(),
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.block0(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x)

# ══════════════════════════════════════════════════════════════════
#  6. MIXUP & CUTMIX AUGMENTATION
# ══════════════════════════════════════════════════════════════════
def mixup_data(x, y, alpha=0.2):
    """Blends two images together with a random ratio."""
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0)).to(device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    """Pastes a random rectangle from one image into another."""
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0)).to(device)

    W, H = x.size(3), x.size(2)
    cut_w = int(W * (1 - lam) ** 0.5)
    cut_h = int(H * (1 - lam) ** 0.5)
    cx = torch.randint(W, (1,)).item()
    cy = torch.randint(H, (1,)).item()

    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, W)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, H)

    x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x, y, y[idx], lam

def mixed_loss(criterion, pred, y_a, y_b, lam):
    """Loss for both MixUp and CutMix — weighted combination."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ══════════════════════════════════════════════════════════════════
#  7. MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════
def run_pipeline():

    # ── Dataset & Stratified Split ────────────────────────────────
    # Stratified ensures every class appears in both train and val
    full_ds = datasets.ImageFolder(DATA_DIR)
    num_classes = len(full_ds.classes)
    print(f"📊 Found {num_classes} classes, {len(full_ds)} total images")

    train_idx, val_idx = train_test_split(
        np.arange(len(full_ds)),
        test_size=0.10,
        stratify=full_ds.targets,    # guarantees balanced split per class
        random_state=SEED
    )

    train_ds = Subset(datasets.ImageFolder(DATA_DIR, train_transforms), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_DIR, val_transforms),   val_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    print(f"   Train: {len(train_ds)} images | Val: {len(val_ds)} images")

    # ── Model + EMA ───────────────────────────────────────────────
    model     = CustomConvNeXt(num_classes).to(device)
    ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    # EMA keeps a smooth running average of all past model weights.
    # Used at inference instead of the raw model for better accuracy.

    # ── Optimizer: each block gets its own learning rate (LLRD) ──
    optimizer = torch.optim.AdamW([
        {'params': model.block0.parameters(), 'lr': 1e-5},   # earliest layers — tiny lr
        {'params': model.block1.parameters(), 'lr': 1e-5},
        {'params': model.block2.parameters(), 'lr': 2e-5},
        {'params': model.block3.parameters(), 'lr': 2e-5},
        {'params': model.head.parameters(),   'lr': 1e-4},   # head — biggest lr
    ], weight_decay=0.05)

    # Cosine schedule that restarts every 5 epochs to escape local minima
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler()
    best_acc  = 0.0

    # ── Training Loop ─────────────────────────────────────────────
    print(f"\n🔥 Training for {EPOCHS} epochs...")
    for epoch in range(EPOCHS):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS}"):
            imgs, lbls = imgs.to(device), lbls.to(device)

            # Randomly choose: MixUp (33%) | CutMix (33%) | Normal (33%)
            r = torch.rand(1).item()
            if r < 0.33:
                imgs, y_a, y_b, lam = mixup_data(imgs, lbls)
                with autocast():
                    loss = mixed_loss(criterion, model(imgs), y_a, y_b, lam)
            elif r < 0.66:
                imgs, y_a, y_b, lam = cutmix_data(imgs, lbls)
                with autocast():
                    loss = mixed_loss(criterion, model(imgs), y_a, y_b, lam)
            else:
                with autocast():
                    out  = model(imgs)
                    loss = criterion(out, lbls)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()

            # ✅ NEW: Gradient clipping — prevents exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            # ✅ NEW: Update EMA weights after every optimizer step
            ema_model.update_parameters(model)

            total_loss += loss.item() * imgs.size(0)
            total      += lbls.size(0)

        scheduler.step()

        # ── Validation (uses EMA model) ───────────────────────────
        ema_model.eval()
        correct, total_val = 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                with autocast():
                    out = ema_model(imgs)
                correct   += (out.argmax(1) == lbls).sum().item()
                total_val += lbls.size(0)

        val_acc = correct / total_val
        avg_loss = total_loss / total
        print(f"  Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Best: {best_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "state_dict":     ema_model.state_dict(),
                "best_acc":       best_acc,
                "num_classes":    num_classes,
                "class_to_idx":   full_ds.class_to_idx,
            }, SAVE_PATH)
            print(f"  ⭐ New best: {best_acc:.4f} — saved checkpoint")

    # ══════════════════════════════════════════════════════════════
    #  8. INFERENCE WITH 4-TRANSFORM TTA
    # ══════════════════════════════════════════════════════════════
    print(f"\n✅ Loading best checkpoint (acc={best_acc:.4f})")
    ckpt = torch.load(SAVE_PATH, map_location=device)
    ema_model.load_state_dict(ckpt["state_dict"])
    ema_model.eval()

    # Fallback label for any corrupted images
    label_counts   = Counter(full_ds.targets)
    dominant_class = label_counts.most_common(1)[0][0]

    test_files      = sorted(glob.glob(os.path.join(TEST_DIR, "*.*")))
    results         = []
    corrupted_count = 0

    print(f"📸 Running inference on {len(test_files)} images with {len(TTA_TRANSFORMS)}-transform TTA...")
    with torch.no_grad():
        for path in tqdm(test_files):
            fname = os.path.basename(path)
            try:
                img    = Image.open(path).convert("RGB")
                logits = 0
                for t in TTA_TRANSFORMS:
                    inp     = t(img).unsqueeze(0).to(device)
                    with autocast():
                        logits += torch.softmax(ema_model(inp), dim=1)
                pred = logits.argmax(1).item()
                results.append({"ImageName": fname, "label": pred})
            except Exception as e:
                corrupted_count += 1
                print(f"⚠️  Corrupted: {fname} → assigned dominant class {dominant_class}")
                results.append({"ImageName": fname, "label": dominant_class})

    if corrupted_count:
        print(f"\n⚠️  {corrupted_count} corrupted images handled.")
    else:
        print("\n✅ No corrupted images.")

    pd.DataFrame(results).to_csv("submission.csv", index=False)
    print("✅ submission.csv created!")
    print(f"\n🏆 Final best validation accuracy: {best_acc:.4f}")

# ══════════════════════════════════════════════════════════════════
if __name__ == '__main__':
    run_pipeline()

🖥️  Using device: cuda
📂 Train: /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train
📂 Test:  /kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test
📊 Found 17 classes, 13163 total images
   Train: 11846 images | Val: 1317 images
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:01<00:00, 204MB/s]


✅ Loaded ConvNeXt-Base ImageNet weights


/tmp/ipykernel_22/2848451122.py:223: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()



🔥 Training for 20 epochs...


...

⚠️  Corrupted: testimage_3427.jpg → assigned dominant class 1


 65%|██████▍   | 3537/5482 [06:14<03:33,  9.10it/s]

⚠️  Corrupted: testimage_4180.jpg → assigned dominant class 1


100%|██████████| 5482/5482 [09:47<00:00,  9.33it/s]



⚠️  4 corrupted images handled.
✅ submission.csv created!

🏆 Final best validation accuracy: 0.5209
